In [98]:
import copy
import os
import pandas as pd
from utilities.data_manager import DataManager
from utilities.backtest_analysis import BacktestAnalysis
from strategies import envelope

In [100]:
bitget = DataManager(name="bitget")

BACKTEST_IGNORED_ATTRIBUTES = [
    "__dict__",
    "__doc__",
    "__module__",
    "__weakref__",
    "bad_trades",
    "best_trade",
    "data",
    "good_trades",
    "trades",
    "wallet",
    "worst_trade",
]


def envelope_backtest(setup):
    ticker = setup["ticker"]
    timeframe = setup["timeframe"]
    start_date = setup["start_date"]
    end_date = setup["end_date"] if "end_date" in setup else None
    strategy_params = setup["strategy_params"]
    leverage = setup["leverage"]

    # On load avec symbole, timeframe, dates,
    ohlcv = bitget.load(ticker, timeframe, start_date, end_date)
    envelopeStrategy = envelope.Strategy(strategy_params, ohlcv)
    envelopeStrategy.run_backtest(
        initial_balance=1000,
        leverage=leverage,
        open_fee_rate=0.0002,
        close_fee_rate=0.0006,
    )
    backtest = BacktestAnalysis(envelopeStrategy)
    return toDataFrame(setup, backtest)



def toDataFrame(setup, backtest):
    backtestData = {
        "avg_fee": backtest.avg_fee,
        "avg_pnl_pct_bad_trades": backtest.avg_pnl_pct_bad_trades,
        "avg_pnl_pct_good_trades": backtest.avg_pnl_pct_good_trades,
        "avg_pnl_pct": backtest.avg_pnl_pct,
        "biggest_fee": backtest.biggest_fee,
        "calmar_ratio": backtest.calmar_ratio,
        "final_balance": backtest.final_balance,
        "global_win_rate": backtest.global_win_rate,
        "hodl_pct": backtest.hodl_pct,
        "initial_balance": backtest.initial_balance,
        "max_drawdown_equity": backtest.max_drawdown_equity,
        "max_drawdown_trades": backtest.max_drawdown_trades,
        "max_lose_streak": backtest.max_lose_streak,
        "max_win_streak": backtest.max_win_streak,
        "mean_bad_trades_duration": backtest.mean_bad_trades_duration,
        "mean_good_trades_duration": backtest.mean_good_trades_duration,
        "mean_trade_duration": backtest.mean_trade_duration,
        "mean_trades_per_day": backtest.mean_trades_per_day,
        "performance_vs_hodl": backtest.performance_vs_hodl,
        "profit_factor": backtest.profit_factor,
        "return_over_max_drawdown": backtest.return_over_max_drawdown,
        "roi": backtest.roi,
        "sharpe_ratio": backtest.sharpe_ratio,
        "sortino_ratio": backtest.sortino_ratio,
        "time_in_position_ratio": backtest.time_in_position_ratio,
        "total_bad_trades": backtest.total_bad_trades,
        "total_fee": backtest.total_fee,
        "total_good_trades": backtest.total_good_trades,
        "total_trades": backtest.total_trades,
    }
    data = copy.deepcopy(setup)
    # flatten strategy params
    del data["strategy_params"]
    data.update(setup["strategy_params"])
    data.update(backtestData)
    data.update(
        {
            "start_date": backtest.wallet.index[0],
            "end_date": backtest.wallet.index[-1],
        }
    )
    data.update({(f"envelope {i+1}", None) for i in range(5)})
    data.update(
        {
            (f"envelope {i+1}", v)
            for i, v in enumerate(setup["strategy_params"]["envelopes"])
        }
    )
    return data


def BacktestsToCSVFile(backtests, path):
    df = pd.DataFrame(backtests)
    # print(df.columns)
    columns = [
        "ticker",
        "timeframe",
        "average_period",
        "average_type",
        "stop_loss_pct",
        "price_jump_pct",
        "leverage",
        "mode",
        "envelope 1",
        "envelope 2",
        "envelope 3",
        "envelope 4",
        "envelope 5",
        "start_date",
        "end_date",
        "initial_balance",
        "final_balance",
        "roi",
        "hodl_pct",
        "global_win_rate",
        "sharpe_ratio",
        "max_drawdown_equity",
        "max_drawdown_trades",
        "total_trades",
        "total_good_trades",
        "total_bad_trades",
        "time_in_position_ratio",
        "avg_pnl_pct",
        "avg_pnl_pct_good_trades",
        "avg_pnl_pct_bad_trades",
        "avg_fee",
        "biggest_fee",
        "calmar_ratio",
        "total_fee",
        "max_lose_streak",
        "max_win_streak",
        "mean_trade_duration",
        "mean_good_trades_duration",
        "mean_bad_trades_duration",
        "mean_trades_per_day",
        # "envelopes",
        # "performance_vs_hodl",
        # "position_size_percentage",
        # "profit_factor",
        # "return_over_max_drawdown",
        # "sortino_ratio",
    ]
    mode = 'a' if os.path.exists(path) else 'w'
    header = not os.path.exists(path)
    df.to_csv(path, columns=columns, index=False, mode=mode, header=header)

In [102]:
# actual dch876, 2024-12-05, short and long
setups = [
    {
        "ticker": "DOGE/USDT:USDT",
        "timeframe": "5m",
        "start_date": "2023-01-01 00:00:00",
        "end_date": "2024-01-01 00:00:00",
        "leverage": 1,
        "strategy_params": {
            "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
            "average_period": 5,
            "envelopes": [0.03, 0.05, 0.07, 0.09],
            "stop_loss_pct": 0.45,
            "price_jump_pct": 0.2,
            "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
            # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
            # 'mode': "long"#, "short", 'both' (default)
            # "mode": "long",
        },
    },
    {
        "ticker": "DOGE/USDT:USDT",
        "timeframe": "5m",
        "start_date": "2024-01-01 00:00:00",
        "end_date": "2025-01-01 00:00:00",
        "leverage": 1,
        "strategy_params": {
            "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
            "average_period": 5,
            "envelopes": [0.03, 0.05, 0.07, 0.09],
            "stop_loss_pct": 0.45,
            "price_jump_pct": 0.2,
            "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
            # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
            # 'mode': "long"#, "short", 'both' (default)
            # "mode": "long",
        },
    },
    {
        "ticker": "DOGE/USDT:USDT",
        "timeframe": "5m",
        "start_date": "2023-01-01 00:00:00",
        # "end_date": "2024-01-01 00:00:00",
        "leverage": 1,
        "strategy_params": {
            "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
            "average_period": 5,
            "envelopes": [0.03, 0.05, 0.07, 0.09],
            "stop_loss_pct": 0.45,
            "price_jump_pct": 0.2,
            "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
            # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
            # 'mode': "long"#, "short", 'both' (default)
            # "mode": "long",
        },
    },
]

backtests = [envelope_backtest(setup) for setup in setups]
BacktestsToCSVFile(backtests, "../backtests.csv")